# The cold-start tax on GBDT deployment

Measures what a gradient-boosted tree model costs to **deploy**, not to **run**,
and shows the two rankings are different.

Runs on a free Colab **CPU** runtime. No GPU, no root, no RAPL.

**The one rule that makes this valid:** every measurement runs in a fresh
subprocess. A notebook kernel that has already run `import lightgbm` has already
paid the import cost, so measuring cold start in a normal cell would return a
misleading near-zero. That is why cell 3 writes `probe.py` to disk and the
benchmark shells out to it. Do not "simplify" that away.

**Persistence:** results are written to Google Drive, checkpointed during the
run, and reloaded automatically if the Colab session dies. Cell 2 handles the
mount. Nothing is lost on a disconnect.

**Time:** smoke test ~3 min. Full run ~30-50 min, resumable.

## 1. Install

In [ ]:
!pip install -q lightgbm xgboost catboost onnxruntime onnxmltools onnxconverter-common scikit-learn pandas matplotlib joblib

import lightgbm, xgboost, catboost, onnxruntime, sklearn, numpy, platform, sys
VERSIONS = dict(python=sys.version.split()[0], platform=platform.platform(),
                machine=platform.machine(), lightgbm=lightgbm.__version__,
                xgboost=xgboost.__version__, catboost=catboost.__version__,
                onnxruntime=onnxruntime.__version__, sklearn=sklearn.__version__,
                numpy=numpy.__version__)
for k, v in VERSIONS.items():
    print(f"{k:12s} {v}")

Copy that version block into the paper. A benchmark without pinned versions is
not reproducible, so record the versions with the results.

## 2. Mount Google Drive and configure

In [ ]:
QUICK = True    # <-- leave True for the smoke test. Set False for the full run.

USE_DRIVE       = True        # persist results to Google Drive
DRIVE_FOLDER    = "coldstart" # created under MyDrive
SYNC_ARTIFACTS  = False       # see note below
CHECKPOINT_EVERY = 25         # push raw.jsonl to Drive every N measurements

import os, json, time, pickle, glob, shutil, warnings, subprocess, sys
import numpy as np, pandas as pd, joblib
warnings.filterwarnings("ignore")

IN_COLAB = os.path.isdir("/content")

DRIVE_ROOT = None
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")          # safe to rerun; no-op if mounted
    DRIVE_ROOT = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
    for d in ("results", "figures", "artifacts", "data"):
        os.makedirs(f"{DRIVE_ROOT}/{d}", exist_ok=True)
    print("Drive folder:", DRIVE_ROOT)
elif USE_DRIVE:
    DRIVE_ROOT = os.path.abspath("./coldstart_drive")
    for d in ("results", "figures", "artifacts", "data"):
        os.makedirs(f"{DRIVE_ROOT}/{d}", exist_ok=True)
    print("not on Colab; using local stand-in for Drive:", DRIVE_ROOT)

# Work on local disk, not on Drive directly. Drive is a network filesystem:
# thousands of small appends to it are slow and occasionally lose writes.
# Local is fast, and we checkpoint to Drive on a schedule instead.
ROOT = "/content/coldstart" if IN_COLAB else "./coldstart"
ART, DATA, RES, FIG = (f"{ROOT}/artifacts", f"{ROOT}/data",
                       f"{ROOT}/results", f"{ROOT}/figures")
for d in (ART, DATA, RES, FIG):
    os.makedirs(d, exist_ok=True)
SEED = 42


def _mirror(src, dst, patterns=("*",)):
    if not src or not dst or not os.path.isdir(src):
        return 0
    os.makedirs(dst, exist_ok=True)
    n = 0
    for pat in patterns:
        for f in glob.glob(os.path.join(src, pat)):
            if os.path.isfile(f):
                # copyfile, not copy2: Drive's FUSE layer rejects some
                # metadata copies and copy2 would raise on them.
                shutil.copyfile(f, os.path.join(dst, os.path.basename(f)))
                n += 1
    return n


def push_to_drive(include_artifacts=None):
    if not DRIVE_ROOT:
        return 0
    inc = SYNC_ARTIFACTS if include_artifacts is None else include_artifacts
    n = _mirror(RES, f"{DRIVE_ROOT}/results")
    n += _mirror(FIG, f"{DRIVE_ROOT}/figures")
    if inc:
        n += _mirror(ART, f"{DRIVE_ROOT}/artifacts")
        n += _mirror(DATA, f"{DRIVE_ROOT}/data")
    return n


def pull_from_drive(include_artifacts=None):
    if not DRIVE_ROOT:
        return 0
    inc = SYNC_ARTIFACTS if include_artifacts is None else include_artifacts
    n = _mirror(f"{DRIVE_ROOT}/results", RES)
    n += _mirror(f"{DRIVE_ROOT}/figures", FIG)
    if inc:
        n += _mirror(f"{DRIVE_ROOT}/artifacts", ART)
        n += _mirror(f"{DRIVE_ROOT}/data", DATA)
    return n


restored = pull_from_drive()
print(f"working in {ROOT} | QUICK = {QUICK} | restored {restored} file(s) from Drive")

_raw = f"{RES}/" + ("raw_smoketest.jsonl" if QUICK else "raw_main.jsonl")
if os.path.exists(_raw):
    print(f"found {sum(1 for _ in open(_raw))} existing measurements; "
          f"the benchmark cell will resume rather than restart")

**Why not write straight to Drive?** Drive is mounted as a network filesystem.
The benchmark appends to `raw.jsonl` after every one of ~900 subprocess launches,
and doing that over FUSE is slow and occasionally drops writes. So the run works
on local disk and checkpoints to Drive every 25 measurements plus at the end.
Worst case you lose 25 measurements, which the resume logic then redoes.

**`SYNC_ARTIFACTS` is off by default.** The trained models total several hundred
MB and copying them to Drive is slower than retraining them. Training is
deterministic (fixed seed), so if the session dies you just rerun the training
cell and get byte-identical artifacts back in a few minutes. Turn it on only if
you want the exact model files archived alongside the paper.

## 3. Write the probe

This cell writes a standalone script. It takes exactly one measurement in one
cold process and prints a single JSON line. It is never imported by the
notebook.

In [ ]:
%%writefile probe.py
"""One measurement, one fresh process. Prints a single JSON object to stdout."""
import argparse, json, os, resource, sys, time

# Thread count must be fixed BEFORE numpy/OpenMP load, or the runtime grabs
# every core and timings stop being comparable across formats.
_THREADS = os.environ.get("CS_THREADS", "1")
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_v] = _THREADS

CLOCK = time.perf_counter


def _unpickle(path, fmt):
    if fmt == "joblib":
        import joblib
        return joblib.load(path)
    import pickle
    with open(path, "rb") as f:
        return pickle.load(f)


def load_lgb(path, fmt):
    t0 = CLOCK(); import lightgbm as lgb; t_imp = CLOCK() - t0
    t0 = CLOCK()
    if fmt == "native":
        fn = lgb.Booster(model_file=path).predict
    else:
        fn = _unpickle(path, fmt).predict
    return fn, t_imp, CLOCK() - t0


def load_xgb(path, fmt):
    t0 = CLOCK(); import xgboost as xgb; t_imp = CLOCK() - t0
    t0 = CLOCK()
    if fmt == "native":
        b = xgb.Booster(); b.load_model(path); fn = b.inplace_predict
    else:
        fn = _unpickle(path, fmt).predict
    return fn, t_imp, CLOCK() - t0


def load_cat(path, fmt):
    t0 = CLOCK(); import catboost; t_imp = CLOCK() - t0
    t0 = CLOCK()
    if fmt == "native":
        m = catboost.CatBoost(); m.load_model(path); fn = m.predict
    else:
        fn = _unpickle(path, fmt).predict
    return fn, t_imp, CLOCK() - t0


def load_onnx(path, fmt):
    t0 = CLOCK(); import onnxruntime as ort; t_imp = CLOCK() - t0
    t0 = CLOCK()
    so = ort.SessionOptions()
    so.intra_op_num_threads = int(_THREADS)
    so.inter_op_num_threads = int(_THREADS)
    sess = ort.InferenceSession(path, so, providers=["CPUExecutionProvider"])
    name = sess.get_inputs()[0].name
    t_load = CLOCK() - t0
    return (lambda X: sess.run(None, {name: X})), t_imp, t_load


def get_loader(loader):
    lib, fmt = loader.split("_", 1)
    if fmt == "onnx":
        return lambda p: load_onnx(p, fmt)
    return {"lgb": lambda p: load_lgb(p, fmt),
            "xgb": lambda p: load_xgb(p, fmt),
            "cat": lambda p: load_cat(p, fmt)}[lib]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--loader", required=True)
    ap.add_argument("--artifact", required=True)
    ap.add_argument("--data", required=True)
    ap.add_argument("--batch", type=int, default=1)
    ap.add_argument("--reps", type=int, default=100)
    a = ap.parse_args()

    t_proc0 = CLOCK()

    # numpy is timed alone: every backend pays it, so it is charged to none.
    t0 = CLOCK(); import numpy as np; t_numpy = CLOCK() - t0

    dtype = np.float32 if a.loader.endswith("onnx") else np.float64
    X_all = np.load(a.data).astype(dtype)
    X = np.ascontiguousarray(X_all[:a.batch])
    if X.shape[0] < a.batch:
        k = int(np.ceil(a.batch / X_all.shape[0]))
        X = np.ascontiguousarray(np.tile(X_all, (k, 1))[:a.batch].astype(dtype))

    predict, t_import, t_load = get_loader(a.loader)(a.artifact)

    # First call: includes lazy allocation and thread-pool spin-up. This is
    # what a real first request pays.
    t0 = CLOCK(); predict(X); t_first = CLOCK() - t0

    times = []
    for _ in range(a.reps):
        t0 = CLOCK(); predict(X); times.append(CLOCK() - t0)
    times.sort(); n = len(times)

    rss_kb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    rss_mb = rss_kb / 1024 if sys.platform != "darwin" else rss_kb / 1048576

    print(json.dumps(dict(
        loader=a.loader, artifact=os.path.basename(a.artifact), batch=a.batch,
        reps=a.reps, threads=int(_THREADS), t_numpy_s=t_numpy,
        t_import_s=t_import, t_load_s=t_load, t_first_pred_s=t_first,
        t_steady_p50_s=times[n // 2],
        t_steady_p95_s=times[min(n - 1, int(0.95 * n))],
        t_steady_min_s=times[0], t_in_process_s=CLOCK() - t_proc0,
        peak_rss_mb=rss_mb)))


if __name__ == "__main__":
    main()

## 4. Train the models and export every format

In [ ]:
from sklearn.datasets import (load_breast_cancer, fetch_california_housing,
                              fetch_covtype)
from sklearn.model_selection import train_test_split


def get_datasets(quick):
    out = {}
    # Tiny binary problem. Its job is to isolate FIXED overhead: with a model
    # this small, whatever time remains is pure import tax.
    d = load_breast_cancer()
    Xtr, Xte, ytr, _ = train_test_split(d.data, d.target, test_size=.3,
                                        random_state=SEED, stratify=d.target)
    out["breast"] = (Xtr, ytr, Xte, "binary", 2)
    if quick:
        return out

    d = fetch_california_housing()
    Xtr, Xte, ytr, _ = train_test_split(d.data, d.target, test_size=.3,
                                        random_state=SEED)
    out["calif"] = (Xtr, ytr, Xte, "regression", 1)

    # 7 classes means the artifact holds 7x the trees, which is where the
    # formats start to separate.
    d = fetch_covtype()
    rng = np.random.RandomState(SEED)
    idx = rng.choice(len(d.data), size=50_000, replace=False)
    X, y = d.data[idx], d.target[idx] - 1
    Xtr, Xte, ytr, _ = train_test_split(X, y, test_size=.3, random_state=SEED,
                                        stratify=y)
    out["covtype"] = (Xtr, ytr, Xte, "multiclass", 7)
    return out


def tree_grid(name, quick):
    if quick:
        return [50]
    return [100, 500] if name == "covtype" else [100, 500, 2000]


def build_model(lib, task, n_trees, n_classes):
    if lib == "lgb":
        import lightgbm as lgb
        kw = dict(n_estimators=n_trees, num_leaves=31, learning_rate=.1,
                  random_state=SEED, n_jobs=-1, verbose=-1)
        return lgb.LGBMRegressor(**kw) if task == "regression" else lgb.LGBMClassifier(**kw)
    if lib == "xgb":
        import xgboost as xgb
        kw = dict(n_estimators=n_trees, max_depth=6, learning_rate=.1,
                  random_state=SEED, n_jobs=-1, tree_method="hist", verbosity=0)
        if task == "regression":
            return xgb.XGBRegressor(**kw)
        if task == "multiclass":
            return xgb.XGBClassifier(objective="multi:softprob",
                                     num_class=n_classes, **kw)
        return xgb.XGBClassifier(**kw)
    from catboost import CatBoostClassifier, CatBoostRegressor
    kw = dict(iterations=n_trees, depth=6, learning_rate=.1, random_seed=SEED,
              verbose=0, allow_writing_files=False)
    return CatBoostRegressor(**kw) if task == "regression" else CatBoostClassifier(**kw)

In [ ]:
def export_onnx(model, tag, lib, n_features):
    p = f"{ART}/{tag}.onnx"
    t0 = time.perf_counter()
    if lib == "cat":
        model.save_model(p, format="onnx")
    else:
        import onnxmltools
        from onnxmltools.convert.common.data_types import FloatTensorType
        itypes = [("input", FloatTensorType([None, n_features]))]
        # zipmap=False keeps classifier output a plain array instead of a list
        # of dicts; leaving it on adds cost that would confound the comparison.
        if lib == "lgb":
            onx = onnxmltools.convert_lightgbm(model, initial_types=itypes,
                                               target_opset=13, zipmap=False)
        else:
            try:
                onx = onnxmltools.convert_xgboost(
                    model, initial_types=itypes, target_opset=13,
                    options={id(model): {"zipmap": False}})
            except TypeError:
                onx = onnxmltools.convert_xgboost(model, initial_types=itypes,
                                                  target_opset=13)
        with open(p, "wb") as f:
            f.write(onx.SerializeToString())
    return p, time.perf_counter() - t0


def export_native(model, tag, lib):
    if lib == "lgb":
        p = f"{ART}/{tag}.lgbtxt"; t0 = time.perf_counter()
        model.booster_.save_model(p)
    elif lib == "xgb":
        p = f"{ART}/{tag}.ubj"; t0 = time.perf_counter()
        model.get_booster().save_model(p)
    else:
        p = f"{ART}/{tag}.cbm"; t0 = time.perf_counter()
        model.save_model(p)
    return p, time.perf_counter() - t0


rows = []
for dname, (Xtr, ytr, Xprobe, task, ncls) in get_datasets(QUICK).items():
    np.save(f"{DATA}/{dname}_X.npy",
            np.ascontiguousarray(Xprobe, dtype=np.float64))
    n_feat = Xtr.shape[1]

    for lib in ["lgb", "xgb", "cat"]:
        for n_trees in tree_grid(dname, QUICK):
            tag = f"{dname}__{lib}__t{n_trees}"
            model = build_model(lib, task, n_trees, ncls)
            t0 = time.perf_counter(); model.fit(Xtr, ytr)
            t_train = time.perf_counter() - t0
            print(f"[train] {tag:32s} {t_train:7.2f}s", flush=True)

            def _pk():
                p = f"{ART}/{tag}.pkl"; t = time.perf_counter()
                with open(p, "wb") as f:
                    pickle.dump(model, f, protocol=pickle.HIGHEST_PROTOCOL)
                return p, time.perf_counter() - t

            def _jl():
                p = f"{ART}/{tag}.joblib"; t = time.perf_counter()
                joblib.dump(model, p, compress=0)
                return p, time.perf_counter() - t

            for fmt, fn in [("pickle", _pk), ("joblib", _jl),
                            ("native", lambda: export_native(model, tag, lib)),
                            ("onnx", lambda: export_onnx(model, tag, lib, n_feat))]:
                try:
                    path, t_exp = fn(); size = os.path.getsize(path)
                    status, err = "ok", ""
                except Exception as e:
                    path, t_exp, size = "", float("nan"), -1
                    status, err = "failed", f"{type(e).__name__}: {e}"[:180]
                rows.append(dict(dataset=dname, lib=lib, fmt=fmt,
                                 n_trees=n_trees, task=task, n_features=n_feat,
                                 loader=f"{lib}_{fmt}", artifact=path,
                                 bytes=size, export_s=t_exp, train_s=t_train,
                                 status=status, error=err))
                print(f"   [{'ok  ' if status=='ok' else 'FAIL'}] {fmt:7s} "
                      f"{size/1e6:8.3f} MB  export {t_exp:6.3f}s {err}", flush=True)

manifest = pd.DataFrame(rows)
manifest.to_csv(f"{RES}/manifest.csv", index=False)
push_to_drive()
print(f"\n{(manifest.status=='ok').sum()}/{len(manifest)} artifacts exported")

A `FAIL` on an ONNX row is data, not a bug. Toolchain fragility across pinned
versions is a legitimate finding for a software engineering paper, so report the
failures rather than quietly dropping them.

## 5. Run the benchmark

Each measurement is a separate process launch. This cell appends to
`results/raw.jsonl` after every single one, so if the Colab session dies you
just rerun this cell and it resumes. Nothing is lost, nothing repeated.

In [ ]:
REPEATS = 2 if QUICK else 5      # cold starts per cell, each a fresh process
REPS    = 100                    # steady-state predictions inside each process
THREADS = 1
BATCHES = [1, 32] if QUICK else [1, 32, 1024]
RAW     = f"{RES}/" + ("raw_smoketest.jsonl" if QUICK else "raw_main.jsonl")

env = dict(os.environ, CS_THREADS=str(THREADS))

# Wall time of a Python process that does nothing: the unavoidable floor.
_f = []
for _ in range(5):
    t0 = time.perf_counter()
    subprocess.run([sys.executable, "-c", "pass"], env=env, check=True)
    _f.append(time.perf_counter() - t0)
FLOOR = sorted(_f)[2]
print(f"bare interpreter start-up floor: {FLOOR*1000:.1f} ms\n")

have = set()
if os.path.exists(RAW):
    for line in open(RAW):
        try:
            r = json.loads(line)
            have.add((r["artifact"], r["loader"], r["batch"], r["repeat"]))
        except Exception:
            pass

man = manifest[manifest.status == "ok"].reset_index(drop=True)
todo = [(r, b, k) for _, r in man.iterrows() for b in BATCHES
        for k in range(REPEATS)
        if (os.path.basename(r.artifact), r.loader, b, k) not in have]
print(f"{len(todo)} measurements to run\n")

t_start = time.perf_counter()
with open(RAW, "a") as out:
    for i, (row, batch, k) in enumerate(todo, 1):
        cmd = [sys.executable, "probe.py", "--loader", row.loader,
               "--artifact", row.artifact, "--data", f"{DATA}/{row.dataset}_X.npy",
               "--batch", str(batch), "--reps", str(REPS)]
        t0 = time.perf_counter()
        try:
            p = subprocess.run(cmd, env=env, capture_output=True, text=True,
                               timeout=300)
        except subprocess.TimeoutExpired:
            print(f"  TIMEOUT {row.loader} b={batch}"); continue
        wall = time.perf_counter() - t0
        if p.returncode != 0:
            print(f"  ERROR {row.loader} b={batch}: "
                  f"{p.stderr.strip().splitlines()[-1][:110]}"); continue
        rec = json.loads(p.stdout.strip().splitlines()[-1])
        rec.update(dataset=row.dataset, lib=row.lib, fmt=row.fmt,
                   n_trees=int(row.n_trees), bytes=int(row.bytes), repeat=k,
                   t_process_wall_s=wall, interpreter_floor_s=FLOOR)
        out.write(json.dumps(rec) + "\n"); out.flush()
        if i % CHECKPOINT_EVERY == 0:
            out.flush(); os.fsync(out.fileno())
            push_to_drive(include_artifacts=False)
        if i % 25 == 0 or i == len(todo):
            rate = (time.perf_counter() - t_start) / i
            print(f"  {i}/{len(todo)}   eta {(len(todo)-i)*rate/60:5.1f} min",
                  flush=True)

push_to_drive()
print("\nwrote", RAW, "| checkpointed to Drive")

## 6. Headline numbers

In [ ]:
df = pd.read_json(RAW, lines=True)
# Each design writes to its own file, so no filtering is needed here.

# Time to first prediction as a user experiences it: process launch to answer.
df["ttfp_s"] = (df.interpreter_floor_s + df.t_numpy_s + df.t_import_s
                + df.t_load_s + df.t_first_pred_s)
df["import_share"] = df.t_import_s / df.ttfp_s

keys = ["dataset", "lib", "fmt", "n_trees", "batch"]
mets = ["bytes", "t_numpy_s", "t_import_s", "t_load_s", "t_first_pred_s",
        "t_steady_p50_s", "t_steady_p95_s", "ttfp_s", "t_process_wall_s",
        "peak_rss_mb", "import_share"]
summary = df.groupby(keys)[mets].median().reset_index()
summary.to_csv(f"{RES}/summary.csv", index=False)
push_to_drive()

b1 = df[df.batch == 1]
print("Share of time-to-first-prediction spent on library import")
for (lib, fmt), v in b1.groupby(["lib", "fmt"]).import_share.median().sort_values(ascending=False).items():
    print(f"   {lib}/{fmt:7s} {v*100:5.1f}%")

print("\nMedian time-to-first-prediction by format")
t = b1.groupby("fmt").ttfp_s.median().sort_values()
for fmt, v in t.items():
    print(f"   {fmt:7s} {v*1000:8.1f} ms  ({v/t.iloc[0]:.1f}x)")

print("\nMedian peak RSS by format")
r = b1.groupby("fmt").peak_rss_mb.median().sort_values()
for fmt, v in r.items():
    print(f"   {fmt:7s} {v:8.1f} MB  ({v/r.iloc[0]:.1f}x)")

print("\nRank flip: is the fastest steady-state format also the fastest cold start?")
for batch in sorted(df.batch.unique()):
    s = df[df.batch == batch]
    fs = s.groupby("fmt").t_steady_p50_s.median().idxmin()
    fc = s.groupby("fmt").ttfp_s.median().idxmin()
    print(f"   batch {batch:5d}: steady={fs:7s} cold={fc:7s}  "
          f"[{'same' if fs == fc else 'FLIP'}]")

The flip line is the one to watch on the full run. If a flip appears at batch
1024, the paper's thesis is "the ranking depends on the serving regime." If no
flip appears anywhere, pivot to the decomposition itself: deployment cost is
90%+ fixed library overhead, which already contradicts the folk wisdom that
native formats load faster than pickle.

## 7. Figures

In [ ]:
import matplotlib.pyplot as plt

PALETTE = {"pickle": "#8c7ae6", "joblib": "#487eb0",
           "native": "#e1b12c", "onnx": "#44bd32"}
ORDER = ["pickle", "joblib", "native", "onnx"]

# Fig 1: where the time actually goes
g = b1.groupby(["lib", "fmt"])[["interpreter_floor_s", "t_numpy_s",
                                "t_import_s", "t_load_s", "t_first_pred_s"]].median()
g = g.reindex([(l, f) for l in ["lgb", "xgb", "cat"] for f in ORDER
               if (l, f) in g.index])
fig, ax = plt.subplots(figsize=(11, 5))
bottom = np.zeros(len(g))
for name, col, c in [("interpreter", "interpreter_floor_s", "#dcdde1"),
                     ("numpy", "t_numpy_s", "#b2bec3"),
                     ("library import", "t_import_s", "#e17055"),
                     ("model load", "t_load_s", "#0984e3"),
                     ("first predict", "t_first_pred_s", "#00b894")]:
    v = g[col].values * 1000
    ax.bar([f"{l}\n{f}" for l, f in g.index], v, bottom=bottom, label=name,
           color=c, edgecolor="white", linewidth=.6)
    bottom += v
ax.set_ylabel("milliseconds")
ax.set_title("Where time-to-first-prediction actually goes (batch = 1)")
ax.legend(frameon=False, ncol=5, loc="upper center", bbox_to_anchor=(.5, -.12))
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig(f"{FIG}/fig1_breakdown.png", dpi=200,
                                bbox_inches="tight"); plt.show()

In [ ]:
# Fig 2: cold start vs artifact size
g2 = b1.groupby(["fmt", "lib", "dataset", "n_trees"]).agg(
    bytes=("bytes", "median"), ttfp=("ttfp_s", "median")).reset_index()
fig, ax = plt.subplots(figsize=(7.5, 5))
for fmt in ORDER:
    s = g2[g2.fmt == fmt]
    if not s.empty:
        ax.scatter(s.bytes / 1e6, s.ttfp * 1000, label=fmt, s=46,
                   color=PALETTE[fmt], alpha=.85, edgecolor="white")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("artifact size (MB, log)")
ax.set_ylabel("time to first prediction (ms, log)")
ax.set_title("Cold start is nearly flat in artifact size")
ax.legend(frameon=False); ax.grid(alpha=.25, which="both")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig(f"{FIG}/fig2_size_vs_cold.png", dpi=200); plt.show()

# Fig 3: memory
g3 = b1.groupby(["lib", "fmt"]).peak_rss_mb.median().unstack()
g3 = g3[[c for c in ORDER if c in g3.columns]]
fig, ax = plt.subplots(figsize=(7.5, 4.5))
g3.plot(kind="bar", ax=ax, color=[PALETTE[c] for c in g3.columns],
        edgecolor="white", width=.78)
ax.set_ylabel("peak RSS (MB)"); ax.set_xlabel("")
ax.set_title("Resident memory of a single-model serving process")
ax.legend(frameon=False); ax.tick_params(axis="x", rotation=0)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig(f"{FIG}/fig3_memory.png", dpi=200); plt.show()

# Fig 4: steady state vs batch
g4 = df.groupby(["fmt", "batch"]).t_steady_p50_s.median().reset_index()
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for fmt in ORDER:
    s = g4[g4.fmt == fmt].sort_values("batch")
    if not s.empty:
        ax.plot(s.batch, s.t_steady_p50_s * 1000, marker="o", label=fmt,
                color=PALETTE[fmt], lw=2)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("batch size (log)")
ax.set_ylabel("median steady-state latency (ms, log)")
ax.set_title("Steady-state latency, once the process is already warm")
ax.legend(frameon=False); ax.grid(alpha=.25, which="both")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig(f"{FIG}/fig4_steady_batch.png", dpi=200); plt.show()

push_to_drive()
print("figures synced to Drive")

## 8. Prediction equivalence check

This check confirms that ONNX and the native format return the same
answers. Without this, the recommendation to switch formats is unsupported.

In [ ]:
import onnxruntime as ort

# CatBoost's ONNX classifier returns a list of {class: prob} dicts. Flatten
# that to a plain array so it can be compared against the native output.
def _to_array(out):
    if isinstance(out, list) and out and isinstance(out[0], dict):
        keys = sorted(out[0].keys())
        return np.array([[float(d[k]) for k in keys] for d in out],
                        dtype=np.float64)
    return np.asarray(out, dtype=np.float64)


checks = []
for _, r in manifest[(manifest.status == "ok") & (manifest.fmt == "onnx")].iterrows():
    nat = manifest[(manifest.dataset == r.dataset) & (manifest.lib == r.lib)
                   & (manifest.n_trees == r.n_trees) & (manifest.fmt == "native")]
    if nat.empty:
        continue
    X = np.load(f"{DATA}/{r.dataset}_X.npy")[:256]
    try:
        sess = ort.InferenceSession(r.artifact, providers=["CPUExecutionProvider"])
        o = sess.run(None, {sess.get_inputs()[0].name: X.astype(np.float32)})
        onnx_out = _to_array(o[-1] if len(o) > 1 else o[0])

        if r.lib == "lgb":
            import lightgbm as lgb
            native_out = lgb.Booster(model_file=nat.iloc[0].artifact).predict(X)
        elif r.lib == "xgb":
            import xgboost as xgb
            b = xgb.Booster(); b.load_model(nat.iloc[0].artifact)
            native_out = b.inplace_predict(X)
        else:
            import catboost
            m = catboost.CatBoost(); m.load_model(nat.iloc[0].artifact)
            native_out = m.predict(X, prediction_type="Probability"
                                   if r.task != "regression" else "RawFormulaVal")

        a = np.asarray(native_out, dtype=np.float64)
        o2 = onnx_out
        if a.ndim == 1 and o2.ndim == 2 and o2.shape[1] == 2:
            o2 = o2[:, 1]
        if o2.ndim == 2 and a.ndim == 2 and o2.shape != a.shape:
            raise ValueError(f"shape mismatch {o2.shape} vs {a.shape}")
        max_abs = float(np.max(np.abs(o2.ravel()[:a.size] - a.ravel())))
        checks.append(dict(dataset=r.dataset, lib=r.lib, n_trees=r.n_trees,
                           max_abs_diff=max_abs, ok=max_abs < 1e-4))
    except Exception as e:
        checks.append(dict(dataset=r.dataset, lib=r.lib, n_trees=r.n_trees,
                           max_abs_diff=float("nan"), ok=False,
                           note=f"{type(e).__name__}: {e}"[:120]))

eq = pd.DataFrame(checks)
eq.to_csv(f"{RES}/equivalence.csv", index=False)
push_to_drive()
print(eq.to_string(index=False))

Report the largest absolute difference in the paper. Small disagreements are
expected because ONNX runs in float32 while the tree libraries predict in
float64. That is a real trade-off worth one paragraph, not something to hide.

## 9. Final sync and verify

Pushes everything to Drive and lists what actually landed there, so you can
confirm before closing the session.

In [ ]:
with open(f"{RES}/versions.json", "w") as f:
    json.dump(VERSIONS, f, indent=2)

n = push_to_drive(include_artifacts=SYNC_ARTIFACTS)
print(f"synced {n} file(s) to Drive\n")

if DRIVE_ROOT:
    total = 0
    for sub in ("results", "figures", "artifacts", "data"):
        files_ = sorted(glob.glob(f"{DRIVE_ROOT}/{sub}/*"))
        if not files_:
            continue
        print(f"{sub}/")
        for f in files_:
            mb = os.path.getsize(f) / 1e6
            total += mb
            print(f"   {os.path.basename(f):28s} {mb:8.3f} MB")
    print(f"\ntotal on Drive: {total:.1f} MB at {DRIVE_ROOT}")
else:
    print("Drive is off; everything is in", ROOT)

### If the session dies

Reconnect and run cells 1, 2, 3, 4, 5. Cell 2 pulls `raw.jsonl` back from Drive,
cell 4 retrains the models deterministically, and cell 5 skips every measurement
already recorded and continues from there. You lose at most the last 25
measurements.

### To start over from scratch

Delete the Drive folder, or run this in a fresh cell:

```python
import shutil
shutil.rmtree(f"{DRIVE_ROOT}/results", ignore_errors=True)
shutil.rmtree(f"{ROOT}/results", ignore_errors=True)
```

## Limitations to declare in the paper

Put these in threats to validity rather than waiting to be asked.

- **Warm page cache.** Dropping the OS page cache needs root, which Colab does
  not give you. These are warm-cache cold starts: the artifact file is already
  in memory, which understates true first-ever load.
- **Single machine.** One hardware and OS configuration. Report the CPU, RAM and
  every library version from cell 1.
- **Threads fixed at 1** for comparability. Multi-threaded serving shifts the
  steady-state numbers but not the import numbers.
- **ONNX is not free.** Conversion can fail, it constrains which models export
  cleanly, and it takes you out of the training library's ecosystem. Present it
  as a trade-off, not a blanket recommendation.
- **Colab is a shared VM.** Timings vary between sessions. Run the full sweep in
  one session and report medians across repeats, not single measurements.